# Python Method 만들기

Python 함수를 Method로 등록하고, AI가 데이터를 선택해 실행하는 흐름을 살펴봅니다.

**함수 작성 → Method 등록 → 데이터 연결 → 실행**

상품별 매출 순위로 기본 흐름을 익히고, OLS 회귀분석으로 여러 변수를 연결해 봅니다.

In [53]:
import sys
from pathlib import Path

import pandas as pd

repo_root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "examples/methods").is_dir()
)
sys.path.insert(0, str(repo_root))

from decision_layer.methods import Column, DataInput, Scope, python_method
from importlib import reload
from examples.methods.authoring import customer_analysis

practice_session = reload(customer_analysis).practice_session

## 1. 상품별 매출 순위

간단한 순위 계산으로 함수 작성부터 등록과 실행까지 살펴봅니다.

### 계산 함수

`value` 열의 순위를 계산하고 `group` 열과 함께 반환합니다.

In [54]:
def rank_values(data, descending=True):
    """값이 큰 순서로 순위를 매깁니다. 같은 값은 같은 순위로 표시합니다."""
    return data[["group", "value"]].assign(
        rank=data["value"].rank(method="min", ascending=not descending).astype(int)
    )


frame = pd.DataFrame({"group": ["A", "B", "C"], "value": [30, 20, 20]})
standalone = rank_values(frame)
standalone

,group,value,rank
0,A,30,1
1,B,20,2
2,C,20,2


### Method로 등록하기

`python_method(...)`로 실행할 함수와 입력 조건을 묶습니다.

- `name`, `version`: Method의 이름과 버전
- `DataInput`: 조회할 데이터의 조건
- `Column`: 데이터에 필요한 열. `measure`는 지표, `dimension`은 차원입니다.

`practice_session()`으로 예제 데이터가 준비된 세션을 만들고, `session.register(...)`로 Method를 등록합니다.

In [55]:
rank_method = python_method(
    rank_values,
    name="my_team.rank",
    version="1.0.0",
    inputs={
        "data": DataInput(
            columns={
                "value": Column(source="measure"),
                "group": Column(source="dimension"),
            },
        ),
    },
)

session = await practice_session()
session.register(rank_method)
print("등록된 Method:", [method.name for method in session.list_methods()])

등록된 Method: ['my_team.rank']


### 데이터를 연결해 분석하기

상품별 매출 순위를 구하도록 입력을 연결합니다.

- `bindings`: `value`에 매출, `group`에 상품을 연결합니다.
- `params`: `descending=True`로 매출이 큰 순서로 순위를 매깁니다.
- `scope`: 조회 기간과 날짜 기준을 지정합니다.

Decision Layer는 상품별 매출을 조회해 `value`, `group` 열이 있는 DataFrame을 만들고 `rank_values`에 전달합니다.

AI가 입력을 선택하고 실행하는 과정은 3장에서 살펴봅니다.

In [56]:
trial = await session.run(
    "my_team.rank",
    bindings={
        "value": "cube://shop/sales/revenue",
        "group": "cube://shop/sales/product",
    },
    params={"descending": True},
    scope=Scope(
        date_range=("2026-09-01", "2026-09-30"),
        time_dimension="cube://shop/sales/date",
    ),
)

trial.result.primary.data

[{'group': 'A', 'value': 30.0, 'rank': 1},
 {'group': 'B', 'value': 20.0, 'rank': 2},
 {'group': 'C', 'value': 20.0, 'rank': 2}]

## 2. 여러 변수를 사용하는 OLS 분석

앞선 순위 계산은 등록과 실행 흐름을 익히기 위한 예제였습니다. 이번에는 여러 변수와 분석 조건을 다루는 OLS 회귀분석을 Method로 만듭니다.

- 데이터 과학자는 검토한 코드를 한 번 등록합니다.
- 사용자와 AI는 필요한 데이터를 연결해 같은 분석을 재사용합니다.

### 계산 함수

고객의 광고 노출 횟수와 구매 금액의 관계를 살펴봅니다. 나이와 이전 구매 금액도 함께 고려합니다.

함수에는 데이터와 분석할 열 이름을 전달합니다.

- `data`: 고객별 DataFrame
- `outcome`: 구매 금액
- `treatment`: 광고 노출 횟수
- `confounders`: 나이, 이전 구매 금액

In [57]:
def ols(data, outcome, treatment, confounders):
    """다른 변수들을 함께 고려해 관심 변수와 결과의 관계를 OLS로 분석합니다."""
    import statsmodels.api as sm

    design = sm.add_constant(data[[treatment, *confounders]], has_constant="add")
    fit = sm.OLS(data[outcome], design).fit()
    return {
        "value": float(fit.params[treatment]),
        "n_observations": int(fit.nobs),
        "r_squared": float(fit.rsquared),
        "coefficients": {key: float(value) for key, value in fit.params.items()},
    }

### 분석 조건 검사하기

`check_ols`는 결측값과 변수 구성 등 계산에 필요한 조건을 확인합니다. `checks=[check_ols]`로 등록하면 계산 전에 실행합니다.

검사 함수는 선택 사항입니다. `ValueError`가 발생하거나 `False`를 반환하면 이유를 기록하고 계산을 멈춥니다.

In [58]:
def check_ols(data, outcome, treatment, confounders):
    """OLS 계산에 필요한 데이터와 변수 구성을 확인합니다."""
    import numpy as np
    import pandas as pd
    import statsmodels.api as sm

    columns = [outcome, treatment, *confounders]
    if len(set(columns)) != len(columns):
        raise ValueError("같은 열을 두 번 선택했습니다.")

    values = data[columns]
    if not all(pd.api.types.is_numeric_dtype(values[col]) for col in columns):
        raise ValueError("숫자형 열만 선택해 주세요.")
    if not np.isfinite(values.to_numpy(dtype=float)).all():
        raise ValueError("결측값이나 무한대를 먼저 처리해 주세요.")
    if values[outcome].nunique() < 2:
        raise ValueError("결과 변수의 값이 모두 같습니다.")

    design = sm.add_constant(values[[treatment, *confounders]], has_constant="add")
    if len(data) <= design.shape[1]:
        raise ValueError(
            "분석할 변수가 많습니다. 데이터를 늘리거나 변수를 줄여 주세요."
        )
    if np.linalg.matrix_rank(design) < design.shape[1]:
        raise ValueError(
            "설명변수들이 서로 완전히 겹칩니다. 변수 구성을 확인해 주세요."
        )

### 직접 실행하기

In [59]:
from examples.methods.authoring.customer_analysis import customer_sample

customers = customer_sample()

check_ols(
    customers,
    outcome="purchase_amount",
    treatment="ad_exposures",
    confounders=["age", "prior_purchase_amount"],
)

direct_ols = ols(
    customers,
    outcome="purchase_amount",
    treatment="ad_exposures",
    confounders=["age", "prior_purchase_amount"],
)

direct_ols

{'value': 50.00000000000206,
 'n_observations': 200,
 'r_squared': 1.0,
 'coefficients': {'const': 999.999999999618,
                  'ad_exposures': 50.00000000000206,
                  'age': 9.999999999999813,
                  'prior_purchase_amount': 0.20000000000000567}}

### Method로 등록하기

- `role="entity_key"`: 고객별 한 행의 기준
- `multiple=True`: 여러 변수 선택

실행 순서는 **데이터 조회 → 기본 입력 검사 → `check_ols` → `ols`**입니다.

기본 입력 검사는 `DataInput`에 정한 조건을 확인합니다.

- `minimum_rows=10`: 데이터가 10행 이상인지
- `columns`: 필요한 열이 모두 있는지
- `dtype="number"`: 숫자형이며 결측값이나 무한대가 없는지
- `role="entity_key"`: 고객 ID가 비어 있거나 중복되지 않는지

검사를 통과한 데이터만 `ols`에 전달합니다.

In [60]:
my_ols = python_method(
    ols,
    name="my_team.ols",
    version="1.0.0",
    inputs={
        "data": DataInput(
            columns={
                "entity_id": Column(source="dimension", role="entity_key"),
                "outcome": Column(source="measure", dtype="number"),
                "treatment": Column(
                    source=["dimension", "measure"],
                    dtype="number",
                ),
                "confounders": Column(
                    source=["dimension", "measure"],
                    dtype="number",
                    multiple=True,
                ),
            },
            minimum_rows=10,
        )
    },
    checks=[check_ols],
    requires_packages={"statsmodels": ">=0.14,<0.16"},
)

session.register(my_ols)
print("등록된 Method:", [method.name for method in session.list_methods()])

등록된 Method: ['my_team.rank', 'my_team.ols']


### 데이터를 연결해 분석하기

In [61]:
ols_trial = await session.run(
    "my_team.ols",
    bindings={
        "entity_id": "cube://shop/customers/id",
        "outcome": "cube://shop/customers/purchase_amount",
        "treatment": "cube://shop/customers/ad_exposures",
        "confounders": [
            "cube://shop/customers/age",
            "cube://shop/customers/prior_purchase_amount",
        ],
    },
    scope=Scope(
        date_range=("2026-02-01", "2026-02-28"),
        time_dimension="cube://shop/customers/date",
    ),
)

ols_trial.result.primary.data

{'value': 50.00000000000206,
 'n_observations': 200,
 'r_squared': 1.0,
 'coefficients': {'const': 999.999999999618,
                  'treatment': 50.00000000000206,
                  'confounders_0': 9.999999999999813,
                  'confounders_1': 0.20000000000000567}}

### 선택한 입력이 함수에 전달되기까지

**1. 입력을 확인합니다**

`bindings`에 연결한 항목이 Cube에 정의되어 있는지, 종류와 자료형이 Method의 입력 조건에 맞는지 확인합니다.

**2. 고객별 데이터를 조회합니다**

선택한 지표와 차원, 기간을 Cube 쿼리로 바꿔 조회합니다. Cube는 모델에 정의된 집계와 조인을 적용해 고객별 데이터를 반환합니다.

`/cubejs-api/v1/load`로 보내는 JSON입니다.

```json
{
  "query": {
    "dimensions": ["customers.id", "customers.age"],
    "measures": [
      "customers.purchase_amount",
      "customers.ad_exposures",
      "customers.prior_purchase_amount"
    ],
    "timeDimensions": [
      {
        "dimension": "customers.date",
        "dateRange": ["2026-02-01", "2026-02-28"]
      }
    ],
    "order": [["customers.id", "asc"]],
    "limit": 50000,
    "offset": 0
  }
}
```

**3. 함수에 데이터를 전달합니다**

조회 결과를 DataFrame으로 만들고, 열 이름을 Method에 등록할 때 정한 이름으로 바꿉니다.

- 구매 금액 → `outcome`
- 광고 노출 횟수 → `treatment`
- 나이, 이전 구매 금액 → 각각 `confounders_0`, `confounders_1`

이 DataFrame과 열 이름을 `ols`에 전달합니다.

```python
ols(
    data=조회한_DataFrame,
    outcome="outcome",
    treatment="treatment",
    confounders=["confounders_0", "confounders_1"],
)
```

## 3. AI로 Method 실행하기

AI가 앞서 만든 Method를 찾아 실행하는 과정을 살펴봅니다.

> 2026년 2월 고객의 광고 노출 횟수와 구매 금액의 관계를 나이와 이전 구매 금액을 고려해 분석해 줘.

**Method 선택 → 입력 확인 → 데이터 선택 → 실행 → 결과 확인**

아래는 AI가 MCP로 보내는 요청과 응답입니다. 응답은 핵심 필드만 표시했습니다.

### Method 선택하기

`list_methods`로 등록된 Method의 이름과 설명을 읽습니다.

```python
await client.call_tool("list_methods", {})
```

```json
{
  "methods": [
    {
      "name": "my_team.rank",
      "description": "값이 큰 순서로 순위를 매깁니다. 같은 값은 같은 순위로 표시합니다."
    },
    {
      "name": "my_team.ols",
      "description": "다른 변수들을 함께 고려해 관심 변수와 결과의 관계를 OLS로 분석합니다."
    }
  ]
}
```

AI는 설명을 보고 OLS를 선택합니다. 설명은 함수의 docstring에서 가져옵니다.

### 입력 조건 확인하기

`describe_method`로 선택한 Method에 필요한 입력을 확인합니다.

```python
await client.call_tool("describe_method", {"name": "my_team.ols"})
```

```json
{
  "name": "my_team.ols",
  "roles": {
    "entity_id": {
      "kind": "entity"
    },
    "outcome": {
      "kind": "measure",
      "data_type": "number"
    },
    "treatment": {
      "accepted_kinds": ["dimension", "measure"],
      "data_type": "number"
    },
    "confounders": {
      "accepted_kinds": ["dimension", "measure"],
      "data_type": "number",
      "multiple": true
    }
  }
}
```

### 데이터 선택하기

`get_semantic_catalog`로 전체 지표와 차원을 확인합니다. Cube `/meta`의 정의를 Decision Layer 형식으로 정리한 목록입니다.

```python
await client.call_tool("get_semantic_catalog", {})
```

```json
{
  "provider": "cube",
  "instance": "shop",
  "objects": [
    {
      "ref": "cube://shop/customers/ad_exposures",
      "kind": "measure",
      "data_type": "number",
      "title": "광고 노출 횟수"
    },
    {
      "ref": "cube://shop/customers/age",
      "kind": "dimension",
      "data_type": "number",
      "title": "나이"
    },
    {
      "ref": "cube://shop/customers/date",
      "kind": "time_dimension",
      "data_type": "time",
      "title": "집계 기준일"
    },
    {
      "ref": "cube://shop/customers/id",
      "kind": "dimension",
      "data_type": "string",
      "title": "고객 ID"
    },
    {
      "ref": "cube://shop/customers/prior_purchase_amount",
      "kind": "measure",
      "data_type": "number",
      "title": "이전 기간 구매 금액"
    },
    {
      "ref": "cube://shop/customers/purchase_amount",
      "kind": "measure",
      "data_type": "number",
      "title": "구매 금액",
      "entity": "cube://shop/customers/id",
      "time_dimension": "cube://shop/customers/date"
    },
    {
      "ref": "cube://shop/sales/date",
      "kind": "time_dimension",
      "data_type": "time",
      "title": "매출 기준일"
    },
    {
      "ref": "cube://shop/sales/product",
      "kind": "dimension",
      "data_type": "string",
      "title": "상품"
    },
    {
      "ref": "cube://shop/sales/revenue",
      "kind": "measure",
      "data_type": "number",
      "title": "매출"
    }
  ]
}
```

AI는 이름과 정의를 보고 입력 조건에 맞는 항목을 선택합니다.

- `entity_id`: 고객 ID
- `outcome`: 구매 금액
- `treatment`: 광고 노출 횟수
- `confounders`: 나이, 이전 기간 구매 금액

### 분석 실행하기

`start_analysis`에 질문, 기간, 확인할 내용(`goals`)을 전달해 Run을 만듭니다.

```python
started = await client.call_tool(
    "start_analysis",
    {
        "question": "2026년 2월 고객의 광고 노출 횟수와 구매 금액의 관계를 나이와 이전 구매 금액을 고려해 분석해 줘",
        "date_range": ["2026-02-01", "2026-02-28"],
        "time_dimension": "cube://shop/customers/date",
        "goals": [
            {
                "id": "relationship",
                "description": "나이와 이전 구매 금액을 고려한 광고 노출과 구매 금액의 관계",
            }
        ],
    },
)
```

응답에는 이번 실행을 구분하는 `run_id`가 담깁니다.

```json
{
  "run_id": "run_example",
  "status": "open",
  "scope": {
    "date_range": [
      "2026-02-01",
      "2026-02-28"
    ],
    "time_dimension": "cube://shop/customers/date"
  }
}
```

`run_step`에 선택한 입력을 전달합니다. 조회 기간은 앞서 만든 Run의 설정을 따릅니다.

```python
step_result = await client.call_tool(
    "run_step",
    {
        "run_id": started.structured_content["run_id"],
        "method": "my_team.ols",
        "bindings": {
            "entity_id": "cube://shop/customers/id",
            "outcome": "cube://shop/customers/purchase_amount",
            "treatment": "cube://shop/customers/ad_exposures",
            "confounders": [
                "cube://shop/customers/age",
                "cube://shop/customers/prior_purchase_amount",
            ],
        },
        "purpose": "나이와 이전 구매 금액을 고려해 광고 노출 횟수와 구매 금액의 관계 확인",
        "goal_ids": ["relationship"],
    },
)
```

`goal_ids`로 이번 계산을 앞서 정한 목표에 연결합니다. Decision Layer가 데이터를 조회하고 함수를 실행합니다.

실행 결과에는 계산값과 검사 기록이 함께 담깁니다.

```json
{
  "run_id": "run_example",
  "step_id": "step_1",
  "status": "success",
  "primary": {
    "type": "estimate",
    "data": {
      "value": 50.0,
      "n_observations": 200
    }
  },
  "validation": [
    {
      "code": "MINIMUM_ROWS",
      "status": "pass"
    },
    {
      "code": "PYTHON_CHECK",
      "status": "pass"
    }
  ],
  "warnings": [
    "Input checks do not establish source completeness, metric equivalence or causal identification."
  ],
  "provenance": {
    "method": "method://my_team/ols@1.0.0",
    "queries": 1,
    "rows_read": 200
  }
}
```

AI는 계산값과 분석한 고객 수, 검사 결과와 경고를 읽고 답변을 작성합니다. 이 결과는 연관성을 보여주며, 광고의 인과 효과를 뜻하지 않습니다.

- `running`: `wait_for_run`으로 결과를 기다립니다.
- `needs_input`: 부족한 입력을 사용자에게 확인합니다.
- `refused`: 거절 이유를 설명합니다.

### 결과 확인하고 결론 남기기

`get_run`으로 저장된 실행 결과를 확인합니다.

```python
await client.call_tool("get_run", {"run_id": "run_example"})
```

```json
{
  "run_id": "run_example",
  "status": "open",
  "steps": [
    {
      "id": "step_1",
      "method": "my_team.ols",
      "result": {
        "status": "success",
        "primary": {
          "type": "estimate",
          "data": {
            "value": 50.0,
            "n_observations": 200
          }
        }
      }
    }
  ]
}
```

`complete_run`으로 결론을 저장하면 Run이 `completed` 상태가 됩니다.

```json
{
  "run_id": "run_example",
  "status": "completed",
  "conclusion": {
    "answer": "나이와 이전 구매 금액을 함께 고려했을 때, 광고 노출 1회 증가와 구매 금액 50 증가가 연관되어 있습니다.",
    "limitations": [
      "광고의 인과 효과를 입증한 결과는 아닙니다."
    ]
  }
}
```

## 프로젝트에서 사용하기

완성한 함수와 등록 코드를 프로젝트에 저장하면 서버에서도 사용할 수 있습니다.

```text
methods/ols/
├── method.py        # 계산 함수와 Method 등록
├── checks.py        # 선택 사항: 검사 함수
└── test_method.py   # 개발 중 실행하는 테스트
```

검사 함수를 분리했다면 `method.py`에서 `from .checks import check_ols`로 가져와 `checks=[check_ols]`에 등록합니다.

테스트는 `decision-layer test`로 실행합니다.

파일 구성과 서버 실행은 [프로젝트에 등록하기](../../ko/guides/method-projects.md)에서 이어서 살펴봅니다.